# Customer Shopping Behavior Analysis

This notebook covers the data preparation and exploratory analysis stage of an end-to-end
analytics project on a retail customer shopping dataset (3,900 customers, 18 attributes:
demographics, purchase details, and shopping habits).

**Goal of this notebook:** get the data into a clean, well-typed, well-documented shape that
can be loaded into a SQL database, then answer a set of business questions about customer
spend, product performance, and discount behavior.

**Workflow:**
1. Load and inspect the raw data
2. Clean column names and handle missing values
3. Engineer a few derived fields needed for later analysis (age group, spend tier, purchase
   frequency in days)
4. Check for outliers before pushing anything into aggregate SQL queries
5. Load the cleaned table into a SQL database


## 1. Load and inspect the data

In [1]:
import pandas as pd

df = pd.read_csv('customer_shopping_behavior.csv')
df.head()


   Customer ID  Age  ... Payment Method Frequency of Purchases
0            1   55  ...          Venmo            Fortnightly
1            2   19  ...           Cash            Fortnightly
2            3   50  ...    Credit Card                 Weekly
3            4   21  ...         PayPal                 Weekly
4            5   45  ...         PayPal               Annually

[5 rows x 18 columns]

In [2]:
df.shape


(3900, 18)

In [3]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Customer ID             3900 non-null   int64  
 1   Age                     3900 non-null   int64  
 2   Gender                  3900 non-null   str    
 3   Item Purchased          3900 non-null   str    
 4   Category                3900 non-null   str    
 5   Purchase Amount (USD)   3900 non-null   int64  
 6   Location                3900 non-null   str    
 7   Size                    3900 non-null   str    
 8   Color                   3900 non-null   str    
 9   Season                  3900 non-null   str    
 10  Review Rating           3863 non-null   float64
 11  Subscription Status     3900 non-null   str    
 12  Shipping Type           3900 non-null   str    
 13  Discount Applied        3900 non-null   str    
 14  Promo Code Used         3900 non-null   str    
 15

18 columns, 3,900 rows, one row per customer. No transaction date or repeat-visit log is
present — `previous_purchases` and `frequency_of_purchases` are the only fields that hint at
behavior over time, and both are static counts/labels rather than timestamped history. That
limits this analysis to a cross-sectional view (comparing customer segments at a point in
time) rather than a time-series or cohort-retention view, so the business questions later in
this project are scoped accordingly.


In [4]:
df.describe()


       Customer ID          Age  ...  Review Rating  Previous Purchases
count  3900.000000  3900.000000  ...    3863.000000         3900.000000
mean   1950.500000    44.068462  ...       3.750065           25.351538
std    1125.977353    15.207589  ...       0.716983           14.447125
min       1.000000    18.000000  ...       2.500000            1.000000
25%     975.750000    31.000000  ...       3.100000           13.000000
50%    1950.500000    44.000000  ...       3.800000           25.000000
75%    2925.250000    57.000000  ...       4.400000           38.000000
max    3900.000000    70.000000  ...       5.000000           50.000000

[8 rows x 5 columns]

## 2. Clean column names

In [5]:
# Standardize to snake_case for consistency once this is loaded into SQL
df.columns = df.columns.str.lower().str.replace(' ', '_')
df = df.rename(columns={'purchase_amount_(usd)': 'purchase_amount'})
df.columns.tolist()


['customer_id', 'age', 'gender', 'item_purchased', 'category', 'purchase_amount', 'location', 'size', 'color', 'season', 'review_rating', 'subscription_status', 'shipping_type', 'discount_applied', 'promo_code_used', 'previous_purchases', 'payment_method', 'frequency_of_purchases']

## 3. Missing values

In [6]:
df.isnull().sum()


customer_id                0
age                        0
gender                     0
item_purchased             0
category                   0
purchase_amount            0
location                   0
size                       0
color                      0
season                     0
review_rating             37
subscription_status        0
shipping_type              0
discount_applied           0
promo_code_used            0
previous_purchases         0
payment_method             0
frequency_of_purchases     0
dtype: int64

`review_rating` is the only column with missing values (37 rows, under 1% of the data).
Dropping these rows would lose otherwise-complete records, so the missing ratings are imputed
using the median rating **within each product category**, rather than a single global median.
Ratings tend to cluster differently by category (a 3.0 means something different for
Outerwear than for Accessories), so a per-category median is a more defensible fill than one
flat number across the whole dataset.


In [7]:
df['review_rating'] = df.groupby('category')['review_rating'].transform(
    lambda x: x.fillna(x.median())
)
df['review_rating'].isnull().sum()


np.int64(0)

## 4. Check for duplicate / redundant columns

Before engineering new fields, it's worth checking whether any existing columns are
actually duplicates of each other in disguise. `discount_applied` and `promo_code_used` look
like they could be measuring the same thing — worth confirming before treating them as two
independent signals in any analysis.


In [8]:
df[['discount_applied', 'promo_code_used']].head(10)


  discount_applied promo_code_used
0              Yes             Yes
1              Yes             Yes
2              Yes             Yes
3              Yes             Yes
4              Yes             Yes
5              Yes             Yes
6              Yes             Yes
7              Yes             Yes
8              Yes             Yes
9              Yes             Yes

In [9]:
(df['discount_applied'] == df['promo_code_used']).all()


np.True_

Confirmed — every single row has identical values in both columns. In this dataset, a promo
code was used if and only if a discount was applied, so they carry no independent
information. Rather than silently dropping this, it's flagged here as an EDA finding: the
business question set later in this project replaces "promo code effectiveness" (which
would just duplicate "discount effectiveness") with a different angle on discount behavior —
specifically, how it relates to subscription status and season.


In [ ]:
df = df.drop('promo_code_used', axis=1)


## 5. Outlier check

Before any aggregation happens in SQL, it's worth checking whether `purchase_amount`,
`review_rating`, or `previous_purchases` contain extreme values that could distort averages
(a handful of very large purchase amounts, for instance, would inflate `AVG(purchase_amount)`
for a whole segment). The IQR method is used here: anything more than 1.5× the interquartile
range beyond Q1/Q3 is flagged.


In [10]:
def iqr_outlier_count(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return ((series < lower) | (series > upper)).sum()

for col in ['purchase_amount', 'review_rating', 'previous_purchases']:
    print(col, '->', iqr_outlier_count(df[col]), 'outliers')


purchase_amount -> 0 outliers
review_rating -> 0 outliers
previous_purchases -> 0 outliers


No outliers in any of the three numeric fields by the IQR rule. This is a clean,
likely-synthetic dataset (every `purchase_amount` falls between \$20 and \$100, for example),
so the averages used in the SQL business questions later on don't need any additional
trimming or winsorizing — worth stating plainly rather than skipping this check.


## 6. Feature engineering

### 6.1 Age group

Customers are split into four age bands using quartiles (`pd.qcut`) rather than fixed-width
bins (e.g. 18–30, 31–45...). Quartiles guarantee a roughly even number of customers in each
group, which makes later comparisons between age groups fairer — a fixed-width bin could
easily leave one bucket with 80% of the customers and another with 5%.


In [11]:
labels = ['Young Adult', 'Adult', 'Middle-aged', 'Senior']
df['age_group'] = pd.qcut(df['age'], q=4, labels=labels)
df[['age', 'age_group']].head(10)


   age    age_group
0   55  Middle-aged
1   19  Young Adult
2   50  Middle-aged
3   21  Young Adult
4   45  Middle-aged
5   46  Middle-aged
6   63       Senior
7   27  Young Adult
8   26  Young Adult
9   57  Middle-aged

In [12]:
df['age_group'].value_counts()


age_group
Young Adult    1028
Middle-aged     986
Senior          944
Adult           942
Name: count, dtype: int64

### 6.2 Purchase frequency in days

`frequency_of_purchases` is a text label ("Weekly", "Monthly", etc.) rather than a number,
which makes it unusable directly in any quantitative comparison or ranking. It's mapped to an
approximate number of days between purchases so it can be sorted, averaged, or compared
numerically later.


In [13]:
frequency_mapping = {
    'Weekly': 7,
    'Bi-Weekly': 14,
    'Fortnightly': 14,
    'Monthly': 30,
    'Every 3 Months': 90,
    'Quarterly': 90,
    'Annually': 365,
}

df['purchase_frequency_days'] = df['frequency_of_purchases'].map(frequency_mapping)
df[['frequency_of_purchases', 'purchase_frequency_days']].head(10)


  frequency_of_purchases  purchase_frequency_days
0            Fortnightly                       14
1            Fortnightly                       14
2                 Weekly                        7
3                 Weekly                        7
4               Annually                      365
5                 Weekly                        7
6              Quarterly                       90
7                 Weekly                        7
8               Annually                      365
9              Quarterly                       90

### 6.3 Spend tier

To support a value-tier analysis later (combining purchase history with how much a customer
actually spends), `purchase_amount` is split into three tiers using the dataset's own
quartile boundaries rather than arbitrary round numbers, for the same fairness reason as the
age grouping above.


In [14]:
df['purchase_amount'].describe()


count    3900.000000
mean       59.764359
std        23.685392
min        20.000000
25%        39.000000
50%        60.000000
75%        81.000000
max       100.000000
Name: purchase_amount, dtype: float64

In [15]:
def spend_tier(amount):
    if amount < 39:
        return 'Low Spend'
    elif amount <= 81:
        return 'Mid Spend'
    else:
        return 'High Spend'

df['spend_tier'] = df['purchase_amount'].apply(spend_tier)
df['spend_tier'].value_counts()


spend_tier
Mid Spend     2003
Low Spend      971
High Spend     926
Name: count, dtype: int64

## 7. Final check before exporting to SQL

In [16]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype   
---  ------                   --------------  -----   
 0   customer_id              3900 non-null   int64   
 1   age                      3900 non-null   int64   
 2   gender                   3900 non-null   str     
 3   item_purchased           3900 non-null   str     
 4   category                 3900 non-null   str     
 5   purchase_amount          3900 non-null   int64   
 6   location                 3900 non-null   str     
 7   size                     3900 non-null   str     
 8   color                    3900 non-null   str     
 9   season                   3900 non-null   str     
 10  review_rating            3900 non-null   float64 
 11  subscription_status      3900 non-null   str     
 12  shipping_type            3900 non-null   str     
 13  discount_applied         3900 non-null   str     
 14  previous_purchases 

In [17]:
df.head()


   customer_id  age gender  ...    age_group purchase_frequency_days  spend_tier
0            1   55   Male  ...  Middle-aged                      14   Mid Spend
1            2   19   Male  ...  Young Adult                      14   Mid Spend
2            3   50   Male  ...  Middle-aged                       7   Mid Spend
3            4   21   Male  ...  Young Adult                       7  High Spend
4            5   45   Male  ...  Middle-aged                     365   Mid Spend

[5 rows x 20 columns]

Data is clean, typed consistently, free of the redundant `promo_code_used` column, and has
the three derived fields (`age_group`, `purchase_frequency_days`, `spend_tier`) needed for the
SQL business questions. Next step is loading this table into a SQL database.


## 8. Load into SQL

Connection code is included for PostgreSQL, MySQL, and MS SQL Server — uncomment whichever
matches your own setup. Credentials are read from environment variables rather than typed
directly into the notebook, so this can be safely pushed to a public repo without leaking
anything.


In [ ]:
import os
from sqlalchemy import create_engine

# --- PostgreSQL ---
# pip install psycopg2-binary sqlalchemy
# engine = create_engine(
#     f"postgresql+psycopg2://{os.environ['DB_USER']}:{os.environ['DB_PASSWORD']}"
#     f"@{os.environ['DB_HOST']}:{os.environ['DB_PORT']}/{os.environ['DB_NAME']}"
# )

# df.to_sql('customer', engine, if_exists='replace', index=False)


In [ ]:
import os
from sqlalchemy import create_engine, text

engine = create_engine(
    f"postgresql+psycopg2://{os.environ['DB_USER']}:"
    f"{os.environ['DB_PASSWORD']}@"
    f"{os.environ['DB_HOST']}:"
    f"{os.environ.get('DB_PORT', '5432')}/"
    f"{os.environ['DB_NAME']}"
)

create_table_sql = """
CREATE TABLE IF NOT EXISTS customer_shopping (
    customer_id INT PRIMARY KEY,
    age INT,
    gender VARCHAR(20),
    item_purchased VARCHAR(100),
    category VARCHAR(100),
    purchase_amount INT,
    location VARCHAR(100),
    size VARCHAR(20),
    color VARCHAR(50),
    season VARCHAR(20),
    review_rating NUMERIC(3,1),
    subscription_status VARCHAR(10),
    shipping_type VARCHAR(50),
    discount_applied VARCHAR(10),
    previous_purchases INT,
    payment_method VARCHAR(50),
    frequency_of_purchases VARCHAR(30),
    age_group VARCHAR(20),
    purchase_frequency_days INT,
    spend_tier VARCHAR(20)
);
"""

with engine.begin() as conn:
    conn.execute(text(create_table_sql))
    conn.execute(text("TRUNCATE TABLE customer_shopping RESTART IDENTITY;"))

df["age_group"]=df["age_group"].astype(str)
df.to_sql("customer_shopping",con=engine,if_exists="append",index=False,method="multi")
print(f"Successfully loaded {len(df)} rows into PostgreSQL.")